In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Speed_of_sound.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Lift__force_.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Evolutionary_history_of_plants.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/James_Webb_Space_Telescope.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Spin_quantum_number.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Medical_ultrasound.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Penrose_process.txt
/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus/Aviation_biofuel.txt

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [3]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
import pandas as pd
import numpy as np
import re
import string

def load_data(train_path='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', 
              test_path='/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'):
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape: {test_df.shape}")
    
    return train_df, test_df

train_df, test_df = load_data()


def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

Train shape: (2000, 8)
Test shape: (500, 7)


In [5]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in text_columns:
    train_df[f'clean_{col}'] = train_df[col].apply(clean_text)
    test_df[f'clean_{col}'] = test_df[col].apply(clean_text)

print("Text cleaning complete. New columns generated.")

Text cleaning complete. New columns generated.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

all_train_text = train_df[[f'clean_{col}' for col in text_columns]].astype(str).agg(' '.join, axis=1)
tfidf.fit(all_train_text)

print(f"TF-IDF Vocabulary size: {len(tfidf.vocabulary_)}")

TF-IDF Vocabulary size: 2865


In [7]:
# View as a dictionary
train_df[['prompt', 'A', 'B', 'C', 'D', 'E']].iloc[0].to_dict()

{'prompt': "Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.",
 'A': "Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",
 'B': 'Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.',
 'C': 'Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.',
 'D': 'Martin Heidegger be

In [8]:
def get_top_3_tfidf(row, vectorizer):
    prompt_text = row['clean_prompt']
    options_text = [row['clean_A'], row['clean_B'], row['clean_C'], row['clean_D'], row['clean_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    prompt_vec = vectorizer.transform([prompt_text])
    options_vec = vectorizer.transform(options_text)
    similarities = cosine_similarity(prompt_vec, options_vec).flatten()
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    return " ".join(top_3_labels)
train_df['tfidf_prediction'] = train_df.apply(lambda row: get_top_3_tfidf(row, tfidf), axis=1)

print("Predictions generated. Example output:")
print(train_df[['id', 'tfidf_prediction']].head(3))

Predictions generated. Example output:
   id tfidf_prediction
0   1            C D B
1   2            C A B
2   3            E D C


In [9]:
def calculate_map_at_3(true_labels, predicted_labels_list):
    scores = []
    
    for true_label, preds in zip(true_labels, predicted_labels_list):
        pred_list = preds.split() 
        
        if true_label in pred_list:
            # Find the rank (1-indexed)
            rank = pred_list.index(true_label) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
            
    return np.mean(scores)

baseline_map3 = calculate_map_at_3(train_df['answer'], train_df['tfidf_prediction'])
print(f"Baseline TF-IDF MAP@3 Score: {baseline_map3:.4f}")

Baseline TF-IDF MAP@3 Score: 0.2387


In [10]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

train_df['tokens_prompt'] = train_df['clean_prompt'].apply(lambda x: x.split())
train_df['tokens_A'] = train_df['clean_A'].apply(lambda x: x.split())
train_df['tokens_B'] = train_df['clean_B'].apply(lambda x: x.split())
train_df['tokens_C'] = train_df['clean_C'].apply(lambda x: x.split())
train_df['tokens_D'] = train_df['clean_D'].apply(lambda x: x.split())
train_df['tokens_E'] = train_df['clean_E'].apply(lambda x: x.split())

all_tokens = pd.concat([
    train_df['tokens_prompt'], train_df['tokens_A'], 
    train_df['tokens_B'], train_df['tokens_C'], 
    train_df['tokens_D'], train_df['tokens_E']
]).tolist()

w2v_model = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)

def get_sentence_embedding(tokens, model, vector_size):
    valid_words = [word for word in tokens if word in model.wv]
    if not valid_words:
        return np.zeros(vector_size)
    

    return np.mean([model.wv[word] for word in valid_words], axis=0)

def get_top_3_w2v(row, model):
    vector_size = model.vector_size
    prompt_vec = get_sentence_embedding(row['tokens_prompt'], model, vector_size).reshape(1, -1)
    
    options_tokens = [row['tokens_A'], row['tokens_B'], row['tokens_C'], row['tokens_D'], row['tokens_E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    similarities = []
    for opt_tokens in options_tokens:
        opt_vec = get_sentence_embedding(opt_tokens, model, vector_size).reshape(1, -1)
        # Handle cases where vectors are all zeros
        if not np.any(prompt_vec) or not np.any(opt_vec):
            sim = 0.0
        else:
            sim = cosine_similarity(prompt_vec, opt_vec)[0][0]
        similarities.append(sim)
        
    top_3_idx = np.argsort(similarities)[::-1][:3]
    top_3_labels = [labels[i] for i in top_3_idx]
    
    return " ".join(top_3_labels)

train_df['w2v_prediction'] = train_df.apply(lambda row: get_top_3_w2v(row, w2v_model), axis=1)

w2v_map3 = calculate_map_at_3(train_df['answer'], train_df['w2v_prediction'])
print(f"Baseline Word2Vec MAP@3 Score: {w2v_map3:.4f}")

Baseline Word2Vec MAP@3 Score: 0.3234


In [11]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_transformer_embedding(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(model.config.hidden_size)

    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    last_hidden_state = outputs.last_hidden_state
    
    attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * attention_mask
    summed = torch.sum(masked_embeddings, dim=1)
    counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
    
    sentence_embedding = summed / counts
    return sentence_embedding.numpy().flatten()

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
import torch
import numpy as np
from tqdm.notebook import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs for parallel processing!")
    model = torch.nn.DataParallel(model)


def get_batched_embeddings(texts, batch_size=128):
    texts = [str(t) if pd.notna(t) else "" for t in texts]
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Batches"):
        batch = texts[i:i + batch_size]
        
        inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_state.size()).float()
        
        masked_embeddings = last_hidden_state * attention_mask
        summed = torch.sum(masked_embeddings, dim=1)
        counts = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
        
        sentence_embeddings = summed / counts
        
        all_embeddings.append(sentence_embeddings.cpu().numpy())
        
    return np.vstack(all_embeddings)

Utilizing 2 GPUs for parallel processing!


In [13]:
emb_prompts = get_batched_embeddings(train_df['clean_prompt'].tolist())

option_embeddings = {}
for opt in ['A', 'B', 'C', 'D', 'E']:
    print(f"Embedding Option {opt}...")
    option_embeddings[opt] = get_batched_embeddings(train_df[f'clean_{opt}'].tolist())

similarities = []
norm_prompts = np.linalg.norm(emb_prompts, axis=1, keepdims=True)

for opt in ['A', 'B', 'C', 'D', 'E']:
    emb_opt = option_embeddings[opt]
    norm_opt = np.linalg.norm(emb_opt, axis=1, keepdims=True)
    
    sim = np.sum(emb_prompts * emb_opt, axis=1, keepdims=True) / np.maximum(norm_prompts * norm_opt, 1e-9)
    similarities.append(sim)

similarities_matrix = np.concatenate(similarities, axis=1)


top_3_idx = np.argsort(similarities_matrix, axis=1)[:, ::-1][:, :3]
labels_array = np.array(['A', 'B', 'C', 'D', 'E'])

train_df['transformer_prediction'] = [" ".join(labels) for labels in labels_array[top_3_idx]]


transformer_map3 = calculate_map_at_3(train_df['answer'], train_df['transformer_prediction'])
print(f"Transformer MAP@3: {transformer_map3:.4f}")

Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option A...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option B...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

...

Embedding Option D...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embedding Option E...


Embedding Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Transformer MAP@3: 0.4081


In [14]:
from transformers import pipeline

zero_shot_classifier = pipeline(
    "zero-shot-classification", 
    model="valhalla/distilbart-mnli-12-3", 
    device=0
)

def predict_zero_shot_sample(row):
    prompt_text = row['prompt']
    options = [row['A'], row['B'], row['C'], row['D'], row['E']]
    labels = ['A', 'B', 'C', 'D', 'E']
    
    option_to_label = {opt: label for opt, label in zip(options, labels)}
    
    result = zero_shot_classifier(prompt_text, candidate_labels=options)
    
    top_3_options = result['labels'][:3]
    top_3_labels = [option_to_label[opt] for opt in top_3_options]
    
    return " ".join(top_3_labels)

sample_preds = train_df.head(5).apply(predict_zero_shot_sample, axis=1)

for idx, (pred, actual) in enumerate(zip(sample_preds, train_df['answer'].head(5))):
    print(f"Row {idx+1} | Predicted Top 3: {pred} | Actual Answer: {actual}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Row 1 | Predicted Top 3: C D E | Actual Answer: B
Row 2 | Predicted Top 3: D A E | Actual Answer: A
Row 3 | Predicted Top 3: B A C | Actual Answer: C
Row 4 | Predicted Top 3: C D E | Actual Answer: B
Row 5 | Predicted Top 3: B A C | Actual Answer: A


# Milestone 3 

In [15]:
!pip install -q langchain-text-splitters langchain-huggingface faiss-cpu langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-ser

In [16]:
import glob
import faiss
from tqdm.notebook import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS_DIR = "/kaggle/input/datasets/ahmed101sahil/mcq-wikipedia-corpus/my_scraped_corpus"
txt_files = glob.glob(os.path.join(CORPUS_DIR, "*.txt"))
print(f"Found {len(txt_files)} text files.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=150)
all_chunks = []

for file_path in tqdm(txt_files, desc="Chunking Files"):
    filename = os.path.basename(file_path).replace('.txt', '')
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
        
    if text.strip():
        chunks = text_splitter.split_text(text)

        for chunk in chunks:
            all_chunks.append(f"Source: {filename}\n{chunk}")

print(f"Generated {len(all_chunks)} chunks for embedding.")


print("Embedding external knowledge base... (This may take a minute or two)")

corpus_embeddings = get_batched_embeddings(all_chunks, batch_size=256) 


dimension = corpus_embeddings.shape[1]  
index = faiss.IndexFlatIP(dimension)    
faiss.normalize_L2(corpus_embeddings)   
index.add(corpus_embeddings)
print(f"FAISS Index loaded with {index.ntotal} vectors.")


def retrieve_real_context(query_text, top_k=3):
    query_vec = get_batched_embeddings([query_text], batch_size=1)
    faiss.normalize_L2(query_vec)
    
    distances, indices = index.search(query_vec, top_k)
    
    retrieved_texts = [all_chunks[idx] for idx in indices[0]]
    return "\n\n".join(retrieved_texts)

def format_rag_mcq(row):
    prompt = row['prompt']
    context = retrieve_real_context(prompt, top_k=3)
    formatted_text = f"Context:\n{context}\n\nQuestion: {prompt}\n\nOptions:\nA: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}\nE: {row['E']}"
    return formatted_text


sample_prompt = train_df.loc[0, 'prompt']
print(f"PROMPT: {sample_prompt}")
print(f"\nRETRIEVED CONTEXT:\n{retrieve_real_context(sample_prompt)}")

Found 533 text files.


Chunking Files:   0%|          | 0/533 [00:00<?, ?it/s]

Generated 70529 chunks for embedding.
Embedding external knowledge base... (This may take a minute or two)


Embedding Batches:   0%|          | 0/276 [00:00<?, ?it/s]

FAISS Index loaded with 70529 vectors.
PROMPT: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]


RETRIEVED CONTEXT:
Source: Heideggerian_terminology
it is revealed as a threefold condition of Being. Time, the present, and the notion of the "eternal", are modes of temporality, which is the way humanity views time. For Heidegger, it is very different from the mistaken view of time as being a linear series of past, present and future. Instead he sees it as being an ecstasy, an outside-of-itself, of futural projections (possibilities) and one's place in history as a part of one's generation. Possibilities, then, are integral to understanding of

Source: F__C__S__Schiller
the reference to Time could not, of course, be recovered, any more than the individuality of Reality can be deduced, when once ignored. The assumption is made that, to express the 'truth' about Reality, its 'thisness,' individuality, change and its immersion in a certain temporal and spatial environment may be neglected, and the timeless validity of a conception is thus substituted for the living, changing and perish

In [17]:
sample_rag_input = format_rag_mcq(train_df.iloc[0])
print("\n--- FORMATTED RAG INPUT ---")
print(sample_rag_input)

Embedding Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- FORMATTED RAG INPUT ---
Context:
Source: Heideggerian_terminology
it is revealed as a threefold condition of Being. Time, the present, and the notion of the "eternal", are modes of temporality, which is the way humanity views time. For Heidegger, it is very different from the mistaken view of time as being a linear series of past, present and future. Instead he sees it as being an ecstasy, an outside-of-itself, of futural projections (possibilities) and one's place in history as a part of one's generation. Possibilities, then, are integral to understanding of

Source: F__C__S__Schiller
the reference to Time could not, of course, be recovered, any more than the individuality of Reality can be deduced, when once ignored. The assumption is made that, to express the 'truth' about Reality, its 'thisness,' individuality, change and its immersion in a certain temporal and spatial environment may be neglected, and the timeless validity of a conception is thus substituted for the living, c

# Milestone 4 

In [18]:
!pip install peft accelerate datasets trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 14.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.3 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cud

In [19]:
import wandb

wandb.init(
    project="smart-mcq-solver",
    name="deberta-v3-small-lora",
    config={
        "architecture": "DeBERTa-v3-small",
        "peft_method": "LoRA",
        "r": 8,
        "lora_alpha": 16,
        "learning_rate": 5e-4,
        "batch_size": 8,
        "epochs": 3
    }
)

In [20]:
wandb.log({
    "baseline_tfidf_map3": baseline_map3,      
    "transformer_minilm_map3": transformer_map3 
})

In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset


MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_mcq_function(examples):
    first_sentences = []
    second_sentences = []
    
    for i in range(len(examples['prompt'])):
        prompt = examples['prompt'][i]
        
     
        first_sentences.append([prompt] * 5)
        
     
        options = [examples['A'][i], examples['B'][i], examples['C'][i], examples['D'][i], examples['E'][i]]
        second_sentences.append(options)
        
 
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=256, 
        padding="max_length"
    )
    

    return {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }


label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train_df['label'] = train_df['answer'].map(label_map)


hf_dataset = Dataset.from_pandas(train_df[['prompt', 'A', 'B', 'C', 'D', 'E', 'label']])
tokenized_dataset = hf_dataset.map(preprocess_mcq_function, batched=True)

print("Dataset formatted for Multiple Choice Fine-Tuning!")
print(f"Sample input shape for 'input_ids': {len(tokenized_dataset[0]['input_ids'])} x {len(tokenized_dataset[0]['input_ids'][0])}")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset formatted for Multiple Choice Fine-Tuning!
Sample input shape for 'input_ids': 5 x 256


In [22]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)


lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    
    target_modules=["query_proj", "value_proj"], 
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS 
)


peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']


training_args = TrainingArguments(
    output_dir="./deberta_lora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=8,  
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    bf16=True,  
    logging_steps=20,
    report_to="wandb",             
    run_name="deberta-v3-lora"      
)


trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer initialized and ready for fine-tuning!")

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight             

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

trainable params: 148,225 || all params: 142,043,906 || trainable%: 0.1044
Trainer initialized and ready for fine-tuning!


In [23]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,3.184082,3.138672
2,2.573535,2.410156
3,2.259375,2.113281


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=339, training_loss=2.790171575405605, metrics={'train_runtime': 159.7694, 'train_samples_per_second': 33.799, 'train_steps_per_second': 2.122, 'total_flos': 1794488878080000.0, 'train_loss': 2.790171575405605, 'epoch': 3.0})

In [24]:
wandb.finish()

baseline_tfidf_map3,▁
eval/loss,█▃▁
eval/runtime,█▁▃
eval/samples_per_second,▁█▆
eval/steps_per_second,▁█▆
train/epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇███
train/global_step,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇███
train/grad_norm,▁▁▂▁▁▂▁▃▂▂▂█▃▄▅▄
train/learning_rate,██▇▇▆▆▅▅▄▄▃▃▂▂▁▁
train/loss,█████▇▇▆▅▄▃▃▂▁▂▁
+1,...
